# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. `[4, 2]`. В группе `a` сумма равна `1 + 3`, в группе `b` — `2`.
2. `[1, 1] [2, 1]`. `count` пропускает пустые значения, а `size` считает все строки.
3. `20.0`. Без `aggfunc` сводная таблица считает среднее: `(10 + 30) / 2`.
4. `2 3`. `inner` оставляет только совпавшие ключи, `left` сохраняет три строки слева.
5. `4`. Ключ `1` встречается справа дважды, поэтому первая строка левой таблицы размножается.

In [1]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [2]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [3]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [4]:
# 1
by_city = df.groupby("city").agg(orders=("order_id", "count"),
                                 cancel_share=("cancel", "mean"),
                                 avg_days=("delivery_days", "mean"),
                                 known_days=("delivery_days", "count"))
by_city["revenue"] = paid.groupby("city")["amount"].sum()
by_city["cancel_share"] = by_city["cancel_share"] * 100
by_city = by_city.sort_values("revenue", ascending=False)
print("cancel_share — отмены в %, avg_days — дни, known_days — число известных сроков, revenue — ₸.")
print(by_city.round(2).to_string())

sh_summary = by_city.loc["Шымкент"]
print(f"Шымкент выделяется отменами ({sh_summary['cancel_share']:.1f}% на {sh_summary['orders']:.0f} заказах) "
      f"и доставкой ({sh_summary['avg_days']:.2f} дня по {sh_summary['known_days']:.0f} известным срокам). "
      "Проверьте работу курьерской службы в этом городе.")

cancel_share — отмены в %, avg_days — дни, known_days — число известных сроков, revenue — ₸.
           orders  cancel_share  avg_days  known_days     revenue
city                                                             
Алматы       2116          7.75      2.00        2053  29144090.0
Астана       1339          8.81      2.02        1299  17884750.0
Шымкент       776         20.62      4.45         754   8602660.0
Караганда     519          9.06      1.94         495   7020200.0
Актобе        387         10.59      2.07         383   5117380.0
Атырау        143          7.69      2.04         138   1893850.0
Шымкент выделяется отменами (20.6% на 776 заказах) и доставкой (4.45 дня по 754 известным срокам). Проверьте работу курьерской службы в этом городе.


## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [5]:
# 2
pc = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="sum")
pc["change_%"] = (pc[10] / pc[9] - 1) * 100
print("Выручка оплаченных, ₸, и изменение октября к сентябрю:")
print(pc.round(2).to_string())

checks = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="mean")
paid_counts = paid.pivot_table(index="category", columns="month", values="order_id", aggfunc="count")
print("Средний чек, ₸:")
print(checks.round(2).to_string())
print("Число оплаченных заказов для каждого среднего чека:")
print(paid_counts.to_string())

category_total = pc[[9, 10]].sum(axis=1).sort_values(ascending=False)
worst_category = pc["change_%"].sort_values().index[0]
print(f"Лидер — {category_total.index[0]} ({category_total.iloc[0]:,.0f} ₸ за два месяца), "
      f"сильнее всего падает категория «{worst_category}» ({pc.loc[worst_category, 'change_%']:.1f}%).")
print(f"Чек Электроники в октябре — {checks.loc['Электроника', 10]:,.2f} ₸ "
      f"на {paid_counts.loc['Электроника', 10]} оплаченных заказах. Результаты совпадают с ДЗ 4.")
print("Сборка таблицы выручки в моём ДЗ 4 занимала 2 строки, через pivot_table — 1 строку, без расчётов и печати.")
print("Для менеджера: следите за падением Одежды, сохраняя внимание к Электронике как главному источнику выручки.")

Выручка оплаченных, ₸, и изменение октября к сентябрю:
month                 9          10  change_%
category                                     
Дом           4997730.0   5125790.0      2.56
Книги         1065300.0   1140910.0      7.10
Красота       2610950.0   2678990.0      2.61
Одежда        9167100.0   8505730.0     -7.21
Электроника  17402330.0  16968100.0     -2.50
Средний чек, ₸:
month              9         10
category                       
Дом          10499.43  10768.47
Книги         4020.00   3815.75
Красота       6187.09   6158.60
Одежда       13520.80  13045.60
Электроника  32772.75  33600.20
Число оплаченных заказов для каждого среднего чека:
month         9    10
category             
Дом          476  476
Книги        265  299
Красота      422  435
Одежда       678  652
Электроника  531  505
Лидер — Электроника (34,370,430 ₸ за два месяца), сильнее всего падает категория «Одежда» (-7.2%).
Чек Электроники в октябре — 33,600.20 ₸ на 505 оплаченных заказах. Результаты 

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [6]:
# 3
by_delivery = df.groupby("delivery_days").agg(orders=("order_id", "count"),
                                            cancel_share=("cancel", "mean"))
print("Срок — дни, доля отмен — %:")
print(pd.DataFrame({"orders": by_delivery["orders"],
                    "cancel_%": by_delivery["cancel_share"] * 100}).round(2).to_string())
print("Заказов с неизвестным сроком, не вошедших в эту таблицу:", df["delivery_days"].isna().sum())
ratio = by_delivery.loc[8, "cancel_share"] / by_delivery.loc[1, "cancel_share"]
print(f"При сроке 8 дней отменяют {by_delivery.loc[8, 'cancel_share']:.1%} из "
      f"{by_delivery.loc[8, 'orders']:.0f} заказов, при 1 дне — {by_delivery.loc[1, 'cancel_share']:.1%} из "
      f"{by_delivery.loc[1, 'orders']:.0f}; отношение — {ratio:.1f} раза.")
print("Для менеджера: в Шымкенте проверьте увеличение сроков до 5–8 дней, с такими сроками связаны более частые отмены.")

Срок — дни, доля отмен — %:
               orders  cancel_%
delivery_days                  
1.0              1557      4.24
2.0              1570      7.96
3.0              1590     13.40
5.0                86     22.09
6.0               120     21.67
7.0                95     34.74
8.0               104     43.27
Заказов с неизвестным сроком, не вошедших в эту таблицу: 158
При сроке 8 дней отменяют 43.3% из 104 заказов, при 1 дне — 4.2% из 1557; отношение — 10.2 раза.
Для менеджера: в Шымкенте проверьте увеличение сроков до 5–8 дней, с такими сроками связаны более частые отмены.


## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [7]:
# 4
# Ошибка 1: в выручку попали отменённые заказы. Берём paid.
# Ошибка 2: без aggfunc сводная таблица считает средний чек. Для выручки нужна сумма.
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["change_%"] = (p[10] / p[9] - 1) * 100
print("Выручка оплаченных, ₸, и изменение, %:")
print(p.round(2).to_string())
print("Число оплаченных заказов по городам и месяцам:")
print(paid.pivot_table(index="city", columns="month", values="order_id", aggfunc="count").to_string())
print(f"Для менеджера: в Шымкенте выручка изменилась на {p.loc['Шымкент', 'change_%']:.1f}%, "
      "поэтому проверьте рост отмен и сроки доставки.")

Выручка оплаченных, ₸, и изменение, %:
month               9          10  change_%
city                                       
Актобе      2576960.0   2540420.0     -1.42
Алматы     14501070.0  14643020.0      0.98
Астана      9034220.0   8850530.0     -2.03
Атырау       923990.0    969860.0      4.96
Караганда   3648520.0   3371680.0     -7.59
Шымкент     4558650.0   4044010.0    -11.29
Число оплаченных заказов по городам и месяцам:
month       9    10
city               
Актобе     170  176
Алматы     959  993
Астана     603  618
Атырау      63   69
Караганда  250  222
Шымкент    327  289
Для менеджера: в Шымкенте выручка изменилась на -11.3%, поэтому проверьте рост отмен и сроки доставки.


## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [8]:
# 5
print("Ключ клиентов уникален:", cu["customer_id"].is_unique)
m = df.merge(cu, on="customer_id", how="left")
print("Строк до и после merge:", len(df), len(m))
print("Заказов без возраста:", m["age"].isna().sum())
m["age_group"] = m["age"] // 10 * 10
by_age = m.groupby("age_group").agg(orders=("order_id", "count"),
                                    cancel_share=("cancel", "mean"))
paid_m = m[m["status"] == "оплачен"]
by_age["paid_orders"] = paid_m.groupby("age_group")["amount"].count()
by_age["mean_check"] = paid_m.groupby("age_group")["amount"].mean()
by_age["median_check"] = paid_m.groupby("age_group")["amount"].median()
by_age["cancel_share"] = by_age["cancel_share"] * 100
print("Отмены в %, средний и медианный чек в ₸, paid_orders — база для обоих чеков:")
print(by_age.round(2).to_string())

age60 = m[m["age_group"] == 60]
print("Возрасты, фактически попавшие в группу 60:", sorted(age60["age"].unique().tolist()))
print(f"В группе 60 — {len(age60)} заказов, из них {by_age.loc[60, 'paid_orders']:.0f} оплаченных; "
      f"средний чек {by_age.loc[60, 'mean_check']:,.2f} ₸, медиана {by_age.loc[60, 'median_check']:,.0f} ₸.")
print("Для менеджера: в группе только 60-летние, а медиана не самая высокая. "
      "Одного высокого среднего на этой небольшой группе недостаточно для отдельной акции.")

Ключ клиентов уникален: True
Строк до и после merge: 5280 5280
Заказов без возраста: 0
Отмены в %, средний и медианный чек в ₸, paid_orders — база для обоих чеков:
           orders  cancel_share  paid_orders  mean_check  median_check
age_group                                                             
10            229          8.73          209    14044.74       10700.0
20           1226          9.62         1108    14130.42       10100.0
30           1241         11.12         1103    15142.34       10670.0
40           1170         11.11         1040    14792.97       10050.0
50           1294          9.20         1175    14671.80       10240.0
60            120         13.33          104    16779.13       10495.0
Возрасты, фактически попавшие в группу 60: [60]
В группе 60 — 120 заказов, из них 104 оплаченных; средний чек 16,779.13 ₸, медиана 10,495 ₸.
Для менеджера: в группе только 60-летние, а медиана не самая высокая. Одного высокого среднего на этой небольшой группе недоста

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [9]:
# 6
orders_per_customer = df.groupby("customer_id").size()
frequency = orders_per_customer.value_counts().sort_index()
print("Число заказов у каждого клиента, включая отменённые:")
print(orders_per_customer.to_string())
print("Сколько клиентов сделали 1, 2, 3… заказа:")
print(frequency.to_string())
repeat_customers = (orders_per_customer > 1).sum()
print(f"Больше одного заказа сделали {repeat_customers} из {len(orders_per_customer)} клиентов "
      f"({repeat_customers / len(orders_per_customer):.1%}).")

# Для именно повторных покупок отдельно считаем только оплаченные заказы.
paid_per_customer = paid.groupby("customer_id").size()
repeat_buyers = (paid_per_customer > 1).sum()
print(f"Больше одной оплаченной покупки сделали {repeat_buyers} из {len(paid_per_customer)} покупателей "
      f"({repeat_buyers / len(paid_per_customer):.1%}).")
months_per_customer = df.groupby("customer_id")["month"].nunique()
print("Заказывали в обоих месяцах:", (months_per_customer == 2).sum(), "из", len(months_per_customer))
no_orders = (~cu["customer_id"].isin(df["customer_id"])).sum()
print("Клиентов из справочника без заказов:", no_orders, "из", len(cu))
print("Для менеджера: выделите повторных покупателей для удержания, а клиентов без заказов — для проверки активации.")

Число заказов у каждого клиента, включая отменённые:
customer_id
C00001     4
C00002     1
C00003     2
C00004     2
C00005     4
C00006     2
C00007     2
C00008     1
C00009     1
C00010     5
C00011     1
C00012     2
C00014     2
C00016     2
C00018     1
C00019     2
C00020     3
C00021     1
C00022     6
C00023     6
C00024     1
C00025     1
C00026     4
C00027     1
C00028     2
C00030     2
C00031     6
C00032     1
C00033     2
C00034     4
C00035     3
C00036     4
C00038     2
C00039     4
C00040     1
C00042     1
C00043     4
C00044     2
C00045     2
C00046     5
C00048     6
C00049     1
C00050     2
C00051     3
C00052     3
C00053     1
C00054     4
C00055     4
C00056     1
C00057     6
C00058     2
C00059     2
C00060     1
C00061     3
C00062     3
C00063     2
C00064     1
C00065     4
C00066     5
C00067     4
C00068     3
C00069     2
C00070     1
C00071     3
C00072     3
C00073     1
C00074     1
C00075     3
C00077     1
C00078     2
C00079     1
C00080     2

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [10]:
# 7
sh = df[df["city"] == "Шымкент"]
sh_sep = sh[sh["month"] == 9]
sh_oct = sh[sh["month"] == 10]
sh_oct_paid = sh_oct[sh_oct["status"] == "оплачен"]
sep_cancel = sh_sep["cancel"].mean()
oct_cancel = sh_oct["cancel"].mean()
oct_check = sh_oct_paid["amount"].mean()
expected_revenue = len(sh_oct) * (1 - sep_cancel) * oct_check
actual_revenue = sh_oct_paid["amount"].sum()
lost_revenue = expected_revenue - actual_revenue

company_revenue = paid.groupby("month")["amount"].sum()
company_drop = company_revenue.loc[9] - company_revenue.loc[10]
company_without_failure = company_revenue.loc[10] + lost_revenue
company_change = company_without_failure / company_revenue.loc[9] - 1
print(f"Отмены сентября: {sep_cancel:.1%} на {len(sh_sep)} заказах; "
      f"октября: {oct_cancel:.1%} на {len(sh_oct)} заказах.")
print(f"Средний чек октября: {oct_check:,.2f} ₸ на {len(sh_oct_paid)} оплаченных заказах.")
print(f"Выручка Шымкента в октябре при сентябрьской доле отмен: {expected_revenue:,.2f} ₸.")
print(f"Фактическая выручка: {actual_revenue:,.2f} ₸; расчётная потеря: {lost_revenue:,.2f} ₸.")
print(f"Падение компании: {company_drop:,.0f} ₸; потеря Шымкента составляет {lost_revenue / company_drop:.1%} от него.")
print(f"При условиях задачи выручка компании в октябре составила бы {company_without_failure:,.2f} ₸ "
      f"({company_change:+.2%} к сентябрю).")
print("Для менеджера: по этому сценарию рост отмен в Шымкенте мог перевести компанию от роста к падению; "
      "проверьте курьерскую службу. Оценка предполагает прежнее число заказов и прежний октябрьский чек.")

Отмены сентября: 8.9% на 359 заказах; октября: 30.7% на 417 заказах.
Средний чек октября: 13,993.11 ₸ на 289 оплаченных заказах.
Выручка Шымкента в октябре при сентябрьской доле отмен: 5,315,005.73 ₸.
Фактическая выручка: 4,044,010.00 ₸; расчётная потеря: 1,270,995.73 ₸.
Падение компании: 823,890 ₸; потеря Шымкента составляет 154.3% от него.
При условиях задачи выручка компании в октябре составила бы 35,690,515.73 ₸ (+1.27% к сентябрю).
Для менеджера: по этому сценарию рост отмен в Шымкенте мог перевести компанию от роста к падению; проверьте курьерскую службу. Оценка предполагает прежнее число заказов и прежний октябрьский чек.


## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [11]:
# 8
rng = np.random.default_rng(7)
channel_counts = cu["channel"].value_counts()
channels = channel_counts.index.to_numpy()
probabilities = (channel_counts / len(cu)).to_numpy()
print("Вероятности групп равны долям каналов среди", len(cu), "клиентов:")
print((channel_counts / len(cu)).round(4).to_string())

random_changes = []
organic_sizes = []
organic_paid_counts = []
for experiment in range(100):
    # choice задаёт группу каждому клиенту; все его заказы остаются в одной группе.
    # Вероятности фиксированы, фактическое число клиентов в группах немного меняется.
    random_labels = rng.choice(channels, size=len(cu), p=probabilities)
    client_groups = pd.Series(random_labels, index=cu["customer_id"])
    sample = paid.copy()
    sample["random_group"] = sample["customer_id"].map(client_groups)
    revenues = sample.pivot_table(index="random_group", columns="month", values="amount", aggfunc="sum")
    random_changes.append((revenues[10] / revenues[9] - 1) * 100)
    organic_sizes.append((random_labels == "органика").sum())
    organic_sample = sample[sample["random_group"] == "органика"]
    organic_paid_counts.append(organic_sample.groupby("month").size())

random_changes = pd.DataFrame(random_changes)
organic_paid_counts = pd.DataFrame(organic_paid_counts)
print("Изменение выручки каждой группы в 100 повторах, %:")
print(random_changes.round(2).to_string(index=False))
# quantile(0.05) и quantile(0.95) дают границы, между которыми лежат центральные 90% результатов.
low = random_changes["органика"].quantile(0.05)
high = random_changes["органика"].quantile(0.95)
print(f"Для случайной органики 5-й и 95-й перцентили: от {low:+.2f}% до {high:+.2f}% (100 повторов).")
print("Клиентов в случайной органике: от", min(organic_sizes), "до", max(organic_sizes))
print("Диапазон числа оплаченных заказов случайной органики по месяцам:")
print(organic_paid_counts.agg(["min", "max"]).to_string())

real_paid = paid.merge(cu, on="customer_id", how="left")
actual = real_paid.pivot_table(index="channel", columns="month", values="amount", aggfunc="sum")
organic_change = (actual.loc["органика", 10] / actual.loc["органика", 9] - 1) * 100
print(f"Реальная органика: {organic_change:+.2f}%; внутри случайного диапазона:", low <= organic_change <= high)
print("Для менеджера: падение органики укладывается в колебания случайных групп такого размера. "
      "Этой цифры недостаточно для вывода о SEO; проверьте сроки доставки и отмены по каналам.")

Вероятности групп равны долям каналов среди 2200 клиентов:
channel
реклама     0.5150
органика    0.3377
рефералы    0.1473


Изменение выручки каждой группы в 100 повторах, %:
 органика  реклама  рефералы
    -2.26    -1.94     -3.91
    -6.71     1.46     -6.04
    -2.85    -2.42     -1.01
     2.05     1.07    -20.50
    -2.27    -1.49     -5.30
    -5.03    -2.83      7.71
     2.78    -3.50     -9.12
    -8.56     0.12      3.70
    -3.93     1.15     -9.95
     0.45    -0.79    -12.14
    -2.50    -2.55     -1.29
    -3.56    -2.05     -0.72
     3.66    -8.22      5.51
    -2.72    -2.37     -1.28
     0.08    -8.20     14.52
    -7.25     2.40     -6.67
    -4.41     1.96    -11.55
    -1.55    -0.43    -10.78
   -10.16     2.13      2.00
     2.56    -2.07    -13.54
   -15.78     2.49     13.47
    -3.96    -2.89      3.28
    -1.84    -5.60      8.15
     1.11    -3.92     -4.42
     1.15    -4.70     -1.99
   -13.59     7.02     -5.49
    -6.03     0.50     -4.81
    -5.12    -6.97     24.82
   -11.32     0.71     10.19
    -5.64    -0.54     -0.82
     0.93    -5.37      1.26
   -10.98     3.16   